# 03 · Generic Tyre Degradation Model (all circuits)

Notebook 01 fits degradation for **one** race. This notebook trains **one model on every dry race in OpenF1**, so it can predict degradation curves for any circuit, including the next race before it happens.

**Pipeline:** download all races → data-quality check → EDA → layered outlier removal → compare 6 models with **leave-circuits-out** cross-validation → hold-out check on the target circuit → predict the target race → save in the same format notebook 02 reads.

**Target:** `Delta` = fuel-corrected lap time minus that driver's baseline pace in that race (so drivers and cars can be pooled).
**Features:** tyre age, compound, race progress (track evolution), circuit descriptors measured from the data (median lap time, speed traps, sector shares) and weather.

In [ ]:
# --- Setup (works locally or on Colab) ---
import sys, os
if "google.colab" in sys.modules and not os.path.exists("f1strat"):
    REPO = "https://github.com/anerudh-k/f1-tyre-strategy"
    !git clone -q {REPO} repo && cp -r repo/* .
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from f1strat import *
COLORS = {"SOFT": "#e10600", "MEDIUM": "#f5c400", "HARD": "#8a8a8a"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
os.makedirs("figures", exist_ok=True); os.makedirs("data", exist_ok=True)

In [ ]:
# --- Config ---
YEARS = (2023, 2024, 2025, 2026)   # seasons to learn from (OpenF1 starts in 2023)
TARGET_YEAR, TARGET_GP = 2026, "Sao Paulo"   # race to predict (accents optional)
TARGET_OVERRIDES = {}   # known-in-advance values, e.g. {"TrackTemp": 48} from the forecast
FUEL_START_KG, SEC_PER_KG = 100.0, 0.03

## 1. Build the dataset
First run downloads every race (≈ 6 API calls each, roughly 10–20 min for 4 seasons). Everything is cached in `data/cache/`, so later runs take seconds.
*Colab tip:* the cache is lost when the runtime resets. Mount Google Drive and point `cache_dir` there to keep it.

In [ ]:
laps, races = build_dataset(YEARS, fuel_start_kg=FUEL_START_KG, sec_per_kg=SEC_PER_KG)
laps.to_pickle("data/generic_laps.pkl"); races.to_pickle("data/generic_races.pkl")
print(f"\n{(races.Status == 'ok').sum()} races loaded, {(races.Status != 'ok').sum()} failed, {len(laps):,} clean laps")

## 2. Data quality check

In [ ]:
# Races that failed to download or have no usable dry laps
races[(races.Status != "ok") | (races.CleanLaps.fillna(0) == 0)][["Year", "Round", "EventName", "Status", "WetLaps"]]

In [ ]:
ok = races[races.Status == "ok"].copy()
ok["KeptPct"] = 100 * ok.CleanLaps / ok.RawLaps
cols = ["RaceId", "TotalLaps", "RawLaps", "MissingLapTime", "CleanLaps", "KeptPct", "SCLaps", "HadSC",
        "Rain", "PitLoss", "MedianLap", "STSpeed", "I2Speed", "TrackTemp"]
print("Median % of laps kept by basic cleaning:", round(ok.KeptPct.median(), 1))
feat_cols = ["MedianLap", "STSpeed", "I1Speed", "I2Speed", "S1Share", "S2Share", "TrackTemp", "AirTemp", "Humidity", "PitLoss"]
print("Missing values per race-level feature:"); print(ok[feat_cols].isna().sum().to_string())
ok[cols].sort_values("KeptPct").round(2)

In [ ]:
laps[FEATURES + [TARGET]].describe().T.round(3)

## 3. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
laps.groupby(["Year", "Compound"]).size().unstack()[list(COLORS)].plot.bar(ax=ax[0], color=COLORS)
ax[0].set(title="Clean laps per compound", ylabel="laps")
ax[1].hist(laps[TARGET], bins=120, range=(-2, 6)); ax[1].set(title="Target: lap delta (s)", xlabel="s")
for c, g in laps.groupby("Compound"):
    b = g.groupby("TyreLife")[TARGET].median()
    ax[2].plot(b.index, b.values, c=COLORS[c], lw=2, label=c)
ax[2].set(title="Median delta vs tyre age (all tracks)", xlabel="Tyre age", ylabel="s", xlim=(0, 45), ylim=(-1.5, 3))
ax[2].legend(); plt.tight_layout(); plt.savefig("figures/eda_overview.png", bbox_inches="tight")

In [ ]:
# Degradation differs a lot by circuit -> per-stint slopes, median per circuit
sl = stint_slopes(laps)
order = sl[sl.Compound == "MEDIUM"].groupby("Circuit").Slope.median().sort_values().index
fig, ax = plt.subplots(figsize=(14, 4.5))
for i, c in enumerate(["SOFT", "MEDIUM", "HARD"]):
    m = sl[sl.Compound == c].groupby("Circuit").Slope.median().reindex(order)
    ax.bar(np.arange(len(m)) + (i - 1) * 0.27, m.values, 0.27, color=COLORS[c], label=c, edgecolor="k", lw=0.3)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=60, ha="right")
ax.set(ylabel="Median stint slope (s/lap)", title="Tyre degradation by circuit")
ax.legend(); plt.tight_layout(); plt.savefig("figures/eda_deg_by_circuit.png", bbox_inches="tight")

In [ ]:
# Which circuit features move with degradation? (race level)
race_deg = sl[sl.Compound == "MEDIUM"].groupby("RaceId").Slope.median().rename("MediumDeg")
rf = ok.set_index("RaceId")[["MedianLap", "STSpeed", "I1Speed", "I2Speed", "S1Share", "S2Share",
                             "TrackTemp", "AirTemp", "Humidity", "PitLoss"]].join(race_deg)
corr = rf.corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=60, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im); ax.set_title("Spearman correlation (one row per race)")
plt.tight_layout(); plt.savefig("figures/eda_correlation.png", bbox_inches="tight")
corr["MediumDeg"].drop("MediumDeg").sort_values()

## 4. Outlier removal
Layered, so every step's effect is visible:
1. **wet races** (inters/wets used): different physics
2. **races with < 200 clean laps**: unreliable circuit features
3. **stints < 5 laps** and **non-physical stint slopes** (|slope| > 0.4 s/lap): stint/compound data errors. Checked *before* step 4 so trimming laps can't hide a bad stint.
4. **lap residual outliers**: robust z-score (median/MAD, not mean/std, so the outliers don't distort the threshold) of each lap's residual from its race/compound trend, \|z\| > 3.5. Catches traffic, mistakes and timing glitches.
5. **tyre ages above the 99th percentile** per compound: too few laps to learn from

In [ ]:
d, report = clean_dataset(laps, races)
fig, ax = plt.subplots(figsize=(7, 3)); ax.barh(report.step[1:], report.removed[1:]); ax.invert_yaxis()
ax.set(xlabel="laps removed", title=f"Outlier removal: {len(laps):,} -> {len(d):,} laps"); plt.tight_layout()

In [ ]:
# Before vs after on the busiest race
rid = d.RaceId.value_counts().index[0]
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for a, src, t in [(ax[0], laps, "before"), (ax[1], d, "after")]:
    g = src[src.RaceId == rid]
    for c, gg in g.groupby("Compound"):
        a.scatter(gg.TyreLife, gg[TARGET], s=6, alpha=0.5, c=COLORS[c], label=c)
    a.set(title=f"{rid}: {t} ({len(g)} laps)", xlabel="Tyre age")
ax[0].set_ylabel("Delta (s)"); ax[0].legend(); plt.tight_layout(); plt.savefig("figures/outliers_before_after.png", bbox_inches="tight")

## 5. Model comparison
Six models, from simplest to most flexible. Every score is from **leave-circuits-out** cross-validation: all years of a circuit sit in the same fold, so the test is always on tracks the model has never seen.

How to read the table:
- **Test_MAE**: error on unseen circuits. This is the number that matters.
- **Overfit_gap** = test − train MAE: large means the model memorises its training data.
- **RandomSplit_MAE**: score with a normal random split. If it's much lower than Test_MAE, the model is learning *which circuit* rather than *why* tyres wear (leakage).
- **Fold_MAE_std**: how much the score changes between folds (stability).

In [ ]:
results, oof = compare_models(d)
results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8)); y = np.arange(len(results))
ax.barh(y - 0.2, results.Train_MAE, 0.4, label="train"); ax.barh(y + 0.2, results.Test_MAE, 0.4, label="unseen circuits")
ax.set_yticks(y); ax.set_yticklabels(results.Model); ax.invert_yaxis(); ax.set(xlabel="MAE (s)")
ax.legend(); plt.tight_layout(); plt.savefig("figures/model_comparison.png", bbox_inches="tight")

In [ ]:
# Overfitting check: decision tree depth vs error
curve = tree_depth_curve(d)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(curve.depth, curve.train_MAE, "o-", label="train"); ax.plot(curve.depth, curve.test_MAE, "o-", label="unseen circuits")
ax.set(xlabel="max_depth", ylabel="MAE (s)", title="Decision tree: complexity vs error"); ax.legend()
plt.savefig("figures/tree_depth_curve.png", bbox_inches="tight"); curve.round(3)

In [ ]:
BEST = results.Model.iloc[0]          # or pick one by name
CV_RMSE = float(results.set_index("Model").loc[BEST, "Test_RMSE"])
model, feats = fit_final(d, BEST)
print("Final model:", BEST)
permutation_importance_table(model, feats, d).round(4)

In [ ]:
# The structured model is interpretable: shared compound curves + a severity factor per circuit
sm, _ = fit_final(d, "Structured (compound curve x track severity)")
display(sm.coef_table().round(4))
sev = ok.assign(Severity=sm.severity(ok)).groupby("Circuit").Severity.mean().sort_values()
fig, ax = plt.subplots(figsize=(12, 3.5)); ax.bar(sev.index, sev.values); ax.axhline(1, c="k", lw=1)
ax.set(ylabel="x average wear", title="Learned track severity"); plt.xticks(rotation=60, ha="right")
plt.tight_layout(); plt.savefig("figures/track_severity.png", bbox_inches="tight")

## 6. Hold-out check on the target circuit
Retrain **without** the target circuit, predict it, and compare with the curve fitted directly to its own laps (what notebook 01 gives). Note the per-circuit fit has no track-evolution term, so it tends to *understate* wear on tyres used late in the race (hards especially).

In [ ]:
try:
    check, own, gen = holdout_check(d, races, TARGET_GP, BEST)
    display(check.round(3))
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for c in own:
        a = np.arange(1, own[c].max_age_seen + 1)
        ax.plot(a, own[c].predict(a), c=COLORS[c], lw=2, label=f"{c} own-data fit")
        if c in gen: ax.plot(a, gen[c].predict(a), c=COLORS[c], lw=2, ls="--", label=f"{c} generic (unseen)")
    ax.set(xlabel="Tyre age", ylabel="Delta (s)", title=f"{TARGET_GP}: generic model vs its own data"); ax.legend(fontsize=8)
    plt.savefig("figures/holdout_target.png", bbox_inches="tight")
except ValueError as e:
    print("Hold-out skipped:", e)

## 7. Predict the target race and save for notebook 02

In [ ]:
profile = track_profile(races, TARGET_GP, **TARGET_OVERRIDES)
print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in profile.items()})
pred = predict_deg_models(model, feats, profile, d, cv_rmse=CV_RMSE)
display(summary_table(pred))
fig, ax = plt.subplots(figsize=(8, 4.5))
for c, m in pred.items():
    a = np.arange(1, m.max_age_seen + 1); ax.plot(a, m.predict(a), c=COLORS[c], lw=2.5, label=f"{c} ({m.deg_per_lap:.3f} s/lap)")
ax.set(xlabel="Tyre age (laps)", ylabel="Predicted delta (s)", title=f"Predicted degradation – {TARGET_GP} {TARGET_YEAR}")
ax.legend(); plt.savefig("figures/predicted_target.png", bbox_inches="tight")

In [ ]:
tag = str(TARGET_GP).lower().replace(" ", "_")
out = f"data/deg_generic_{TARGET_YEAR}_{tag}.json"
save_models(pred, out, meta=dict(year=TARGET_YEAR, gp=TARGET_GP, event=f"{TARGET_GP} (generic model)",
            total_laps=profile["TotalLaps"], pit_loss_est=profile["PitLoss"], p_sc=profile["P_SC"],
            model=BEST, cv_rmse=CV_RMSE, trained_on=list(YEARS)))
print("saved", out, "-> set MODEL_FILE to this in notebook 02")

## Findings
*(Your own words after running on real data.)*
- Which model generalised best to unseen circuits, and by how much it beat the compound-only baseline.
- Overfitting: train/test gap, and what the tree-depth curve shows.
- Which circuit features drive wear (importance table / severity weights).
- Hold-out: how close the generic curve came to the target circuit's own data.
- Limitations: Pirelli brings different compound hardness (C1–C5) to each track but OpenF1 only reports SOFT/MEDIUM/HARD; extreme circuits (e.g. Monaco) are extrapolation; 2026 is a new regulation era, so compare training on 2026 only vs all seasons.